In [ ]:
# -*- coding: utf-8 -*-
import re, time
from pathlib import Path

import numpy as np
import pandas as pd
import yfinance as yf

# Optional: try to use yahooquery for name-based fallback search
try:
    from yahooquery import search as yq_search
    YQ_OK = True
except Exception:
    YQ_OK = False

# ===================== CONFIG =====================
HOLDINGS_CSV = "/Users/nityaarya/Downloads/Project/blackrock-esg-etf-study/Data/Final Data/holdings_2025_final.csv"

OUT_DIR = Path("/Users/nityaarya/Downloads/Project/blackrock-esg-etf-study/Data/Data for Dashboard/Analysis 3")
OUT_DIR.mkdir(parents=True, exist_ok=True)

PRICES_OUT = OUT_DIR / "holdings_with_prices.csv"
NO_PRICES_OUT = OUT_DIR / "holdings_needing_manual_check.csv"

# Historical range
START_DATE = "2024-01-01"
END_DATE = None               # up to latest

# polite throttling + reliability
TIMEOUT = 30
SLEEP_EACH = 0.05
RETRY = 1                     # extra retry for download
# ===================================================

def nz(s): return "" if pd.isna(s) else str(s).strip()
def up(s): return nz(s).upper()
def is_numeric(s): return s.isdigit()
def rm_ws_punct(s): return re.sub(r"[ \t/]", "", s)

# Filter out obvious non-equities we should mark for manual check early
CURRENCY_SET = {
    "CASH","USD","EUR","GBP","JPY","CNH","CNY","CAD","HKD","AUD","TWD","INR",
    "KRW","CHF","SEK","NOK","DKK","ZAR","TRY","MXN","BRL","PLN","HUF","MYR",
    "NZD","PHP","AED","CLP","CZK","THB","RUB"
}
FUT_RX = re.compile(r"^[A-Z]{1,3}[FGHJKMNQUVXZ]\d{1,2}$")  # e.g., ESU5

# exchange/location -> Yahoo suffix
SUFFIX_MAP = {
    # US
    "NASDAQ": "", "NYSE": "", "NYSE ARCA": "", "NYSE MKT": "",
    # Canada
    "TSX": ".TO", "TORONTO": ".TO", "TSXV": ".V",
    # UK
    "LSE": ".L", "LONDON": ".L",
    # Switzerland
    "SIX": ".SW", "SWISS": ".SW",
    # Germany
    "XETRA": ".DE", "DEUTSCHE BOERSE": ".DE", "FRANKFURT": ".F",
    # Italy
    "BORSA ITALIANA": ".MI", "MILAN": ".MI",
    # Spain
    "BME": ".MC", "MADRID": ".MC",
    # Euronext
    "EURONEXT PARIS": ".PA", "PARIS": ".PA",
    "EURONEXT AMSTERDAM": ".AS", "AMSTERDAM": ".AS",
    "EURONEXT BRUSSELS": ".BR", "BRUSSELS": ".BR",
    "EURONEXT LISBON": ".LS", "LISBON": ".LS",
    "EURONEXT DUBLIN": ".IR", "DUBLIN": ".IR",
    # Nordics
    "STOCKHOLM": ".ST", "NASDAQ STOCKHOLM": ".ST",
    "HELSINKI": ".HE", "NASDAQ HELSINKI": ".HE",
    "COPENHAGEN": ".CO", "NASDAQ COPENHAGEN": ".CO",
    "OSLO": ".OL", "OSE": ".OL",
    "VIENNA": ".VI",
    # APAC
    "ASX": ".AX", "AUSTRALIAN SECURITIES EXCHANGE": ".AX",
    "TSE": ".T", "TOKYO": ".T",
    "HKEX": ".HK", "HONG KONG": ".HK",
    "SGX": ".SI", "SINGAPORE": ".SI",
    "KRX": ".KS", "KOSPI": ".KS", "KOSDAQ": ".KQ",
    # India
    "NSE": ".NS", "NATIONAL STOCK EXCHANGE OF INDIA": ".NS",
    "BSE": ".BO", "BOMBAY": ".BO",
    # LatAm
    "BMV": ".MX", "MEXICO": ".MX",
    "B3": ".SA", "SAO PAULO": ".SA",
}

ALT_SUFFIXES = {
    ".SW": [".SW", ".VX"], ".DE": [".DE", ".F"], ".F": [".F", ".DE"],
    ".TO": [".TO", ".CN"], ".KS": [".KS", ".KQ"], ".KQ": [".KQ", ".KS"],
    ".NS": [".NS", ".BO"], ".BO": [".BO", ".NS"],
    "": [""], ".L": [".L"], ".PA": [".PA"], ".AS": [".AS"], ".BR": [".BR"],
    ".LS": [".LS"], ".IR": [".IR"], ".MI": [".MI"], ".MC": [".MC"],
    ".ST": [".ST"], ".HE": [".HE"], ".CO": [".CO"], ".OL": [".OL"], ".VI": [".VI"],
    ".AX": [".AX"], ".T": [".T"], ".HK": [".HK"], ".SI": [".SI"],
    ".V": [".V"], ".MX": [".MX"], ".SA": [".SA"], ".CN": [".CN"]
}

def infer_suffix(exch, loc):
    e, l = up(exch), up(loc)
    if e in SUFFIX_MAP: return SUFFIX_MAP[e]
    for k, s in SUFFIX_MAP.items():
        if k in e: return s
    for k, s in SUFFIX_MAP.items():
        if k in l: return s
    return ""

def lse_cleanup(sym):
    s = sym.rstrip(".")           # 'BP.' -> 'BP'
    s = s.replace(".", "-")       # 'BT.A' -> 'BT-A'
    return s

def tsx_cleanup(sym):
    return sym.replace(".", "-")  # 'RCI.B' -> 'RCI-B'

def us_shareclass(sym):
    return sym.replace(".", "-")  # 'BRK.B' -> 'BRK-B'

def make_candidates(raw_sym, exch, loc):
    base = rm_ws_punct(nz(raw_sym))
    if not base:
        return []
    U = up(base)
    if U in CURRENCY_SET: return []
    if FUT_RX.match(U): return []

    sfx = infer_suffix(exch, loc)
    cands = []

    if sfx == ".L":  # LSE
        b1 = lse_cleanup(base)
        cands.append(b1 + sfx)
    elif sfx in (".TO", ".V"):  # Canada
        b1 = tsx_cleanup(base)
        cands.append(b1 + sfx)
    elif sfx == "":  # US
        b1 = us_shareclass(base)
        cands.append(b1)
    else:
        # JP/HK numerics
        if is_numeric(base) and sfx in (".T", ".HK"):
            cands.append(str(int(base)) + sfx)
        else:
            cands.append(base + sfx)

    # Try alternates for the market
    for alt in ALT_SUFFIXES.get(sfx, []):
        if alt == sfx: continue
        if is_numeric(base) and alt in (".T", ".HK"):
            cands.append(str(int(base)) + alt)
        else:
            cands.append(base.replace(".", "-") + alt)

    # Raw last (might already have a suffix in data)
    if base not in cands:
        cands.append(base)

    # Dedup
    out, seen = [], set()
    for c in cands:
        if c and c not in seen:
            seen.add(c); out.append(c)
    return out

def quick_has_data(sym):
    """Fast check: small history window to validate a candidate."""
    try:
        df = yf.download(sym, period="5d", interval="1d", progress=False,
                         group_by=None, threads=False, timeout=TIMEOUT, auto_adjust=False)
        return (isinstance(df, pd.DataFrame) and not df.empty and "Close" in df.columns and df["Close"].notna().any())
    except Exception:
        return False

def resolve_symbol(row):
    raw = nz(row.get("company_ticker"))
    exch = row.get("Exchange","")
    loc  = row.get("Location","")
    name = nz(row.get("name_normalized") or row.get("Name"))

    # 1) direct candidates
    for sym in make_candidates(raw, exch, loc):
        if quick_has_data(sym):
            return sym, "direct_ok"

    # 2) dot->dash fallback (if not tried)
    if "." in raw:
        alt = raw.replace(".", "-")
        if quick_has_data(alt):
            return alt, "dot_to_dash_ok"

    # 3) yahooquery name search (optional)
    if YQ_OK and name:
        try:
            res = yq_search(name)
            if isinstance(res, dict) and "quotes" in res:
                for q in res["quotes"]:
                    sym = q.get("symbol")
                    qt  = up(q.get("quoteType",""))
                    if sym and qt in {"EQUITY","ETF"} and quick_has_data(sym):
                        return sym, "yq_name_ok"
        except Exception:
            pass

    return "", "no_working_symbol"

def download_prices(sym):
    """Return (df, ok, note)."""
    for attempt in range(RETRY+1):
        try:
            df = yf.download(sym, start=START_DATE, end=END_DATE, progress=False,
                             group_by=None, threads=False, timeout=TIMEOUT, auto_adjust=False)
            if isinstance(df, pd.DataFrame) and not df.empty and df["Close"].notna().any():
                return df.reset_index(), True, "ok"
            note = "no_data"
        except Exception as e:
            note = f"error:{type(e).__name__}"
        time.sleep(0.25)
    return pd.DataFrame(), False, note

# ---------- Load holdings & prepare unique set ----------
raw_df = pd.read_csv(HOLDINGS_CSV)

cols = [c for c in ["company_ticker","Exchange","Location","Name","name_normalized"] if c in raw_df.columns]
if "company_ticker" not in cols:
    raise ValueError("company_ticker column not found in holdings file.")

u = (
    raw_df[cols]
    .dropna(subset=["company_ticker"])
    .assign(company_ticker=lambda d: d["company_ticker"].astype(str).str.strip())
)
u = u[u["company_ticker"]!=""].drop_duplicates(subset=["company_ticker","Exchange","Location"])

total_unique = len(u)
print(f"Unique holdings to process: {total_unique}")

# ---------- Resolve each to a Yahoo symbol ----------
resolved_rows = []
for i, row in u.reset_index(drop=True).iterrows():
    raw = nz(row["company_ticker"])
    sym, how = resolve_symbol(row)
    resolved_rows.append({
        "raw_company_ticker": raw,
        "Exchange": nz(row.get("Exchange","")),
        "Location": nz(row.get("Location","")),
        "Name": nz(row.get("Name","")),
        "name_normalized": nz(row.get("name_normalized","")),
        "resolved_symbol": sym,
        "resolution_method": how
    })
    if (i+1) % 200 == 0:
        print(f"Resolved {i+1}/{total_unique}…")
    time.sleep(SLEEP_EACH)

resolved_df = pd.DataFrame(resolved_rows)

# ---------- Split into resolved vs unresolved ----------
resolved_ok = resolved_df[resolved_df["resolved_symbol"]!=""].copy()
need_manual = resolved_df[resolved_df["resolved_symbol"]==""].copy()

# ---------- Download prices for resolved ----------
price_parts = []
status_rows = []

for i, r in resolved_ok.iterrows():
    sym = r["resolved_symbol"]
    raw = r["raw_company_ticker"]
    px, ok, note = download_prices(sym)
    if ok:
        px["Ticker"] = sym
        px["Raw"] = raw
        # keep standard columns
        cols_out = ["Date","Open","High","Low","Close","Adj Close","Volume","Ticker","Raw"]
        for c in cols_out:
            if c not in px.columns:
                px[c] = np.nan
        price_parts.append(px[cols_out])
        status_rows.append({"raw_company_ticker": raw, "yahoo_ticker": sym, "status":"success", "note": "priced_ok"})
    else:
        # move to manual list (price failed)
        need_manual = pd.concat([need_manual, r.to_frame().T], ignore_index=True)
        status_rows.append({"raw_company_ticker": raw, "yahoo_ticker": sym, "status":"fail", "note": note})
    if (i+1) % 200 == 0:
        print(f"Downloaded {i+1}/{len(resolved_ok)}…")
    time.sleep(SLEEP_EACH)

# ---------- Write the two required files ----------
prices_out = pd.concat(price_parts, ignore_index=True) if len(price_parts) else pd.DataFrame(
    columns=["Date","Open","High","Low","Close","Adj Close","Volume","Ticker","Raw"]
)
prices_out.to_csv(PRICES_OUT, index=False)

# For dashboard simplicity, keep a compact “manual check” table
manual_cols = ["raw_company_ticker","Exchange","Location","Name","name_normalized","resolved_symbol","resolution_method"]
manual_out = need_manual[manual_cols].drop_duplicates()
manual_out.to_csv(NO_PRICES_OUT, index=False)

# ---------- Coverage report ----------
N_raw = total_unique
N_priced_holdings = prices_out["Raw"].nunique() if not prices_out.empty else 0
N_manual = manual_out["raw_company_ticker"].nunique() if not manual_out.empty else 0

print("\n=== Coverage ===")
print(f"Unique holdings in file      : {N_raw}")
print(f"Holdings with prices         : {N_priced_holdings}")
print(f"Holdings needing manual check: {N_manual}")
print(f"Check sums to total? {N_priced_holdings + N_manual == N_raw}")
print("\nSaved:")
print("  With prices ->", PRICES_OUT)
print("  Manual check ->", NO_PRICES_OUT)


Unique holdings to process: 3035



1 Failed download:
['XTSLA']: YFPricesMissingError('possibly delisted; no price data found  (period=5d) (Yahoo error = "No data found, symbol may be delisted")')

1 Failed download:
['SGAFT']: YFPricesMissingError('possibly delisted; no price data found  (period=5d) (Yahoo error = "No data found, symbol may be delisted")')


Resolved 200/3035…



1 Failed download:
['SKX']: YFPricesMissingError('possibly delisted; no price data found  (period=5d)')


Resolved 400/3035…



1 Failed download:
['AMED']: YFPricesMissingError('possibly delisted; no price data found  (period=5d)')

1 Failed download:
['ZZ_GR289']: YFPricesMissingError('possibly delisted; no price data found  (period=5d) (Yahoo error = "No data found, symbol may be delisted")')

1 Failed download:
['UAC-VI']: YFPricesMissingError('possibly delisted; no price data found  (period=5d) (Yahoo error = "No data found, symbol may be delisted")')

1 Failed download:
['UAC-VI']: YFPricesMissingError('possibly delisted; no price data found  (period=5d) (Yahoo error = "No data found, symbol may be delisted")')


Resolved 600/3035…



1 Failed download:
['BHLB']: YFPricesMissingError('possibly delisted; no price data found  (period=5d)')

1 Failed download:
['MOGA']: YFPricesMissingError('possibly delisted; no price data found  (period=5d) (Yahoo error = "No data found, symbol may be delisted")')

1 Failed download:
['WBA']: YFPricesMissingError('possibly delisted; no price data found  (period=5d)')


Resolved 800/3035…



1 Failed download:
['PPBI']: YFPricesMissingError('possibly delisted; no price data found  (period=5d)')


In [5]:
import pandas as pd, numpy as np, re
from pathlib import Path

BASE = Path("/Users/nityaarya/Downloads/Project/blackrock-esg-etf-study/Data/Final Data")
PAST = BASE / "Combined Past Holdings"
OUT = BASE / "all_holdings_2017_2025.csv"

def to_float(s):
    if s is None:
        return np.nan
    t = str(s).strip()
    if t in {"", "nan", "NA", "N/A", "-", "--", "—"}:
        return np.nan
    t = t.replace("(", "-").replace(")", "")
    t = re.sub(r"[^0-9.\-]", "", t)
    if t.count(".") > 1:
        first, *rest = t.split(".")
        t = first + "." + "".join(rest)
    try:
        return float(t)
    except:
        return np.nan

def load_soi(path):
    df = pd.read_csv(path)
    df.columns = [c.strip().lower() for c in df.columns]
    req = {"etf_ticker","etf_name","company_ticker","name_normalized","value"}
    if not req.issubset(df.columns):
        raise ValueError(f"Missing columns in {path.name}: {req - set(df.columns)}")
    df["value"] = df["value"].map(to_float).fillna(0.0)
    if "total_overall_value" in df.columns:
        df["total_overall_value"] = df["total_overall_value"].map(to_float)
    y = re.search(r"soi_(\d{4})_final\.csv$", path.name, flags=re.I)
    if not y:
        raise ValueError(f"Year parse failed for {path.name}")
    year = int(y.group(1))
    gsum = df.groupby("etf_ticker")["value"].transform("sum")
    if "total_overall_value" in df.columns:
        tv = df.groupby("etf_ticker")["total_overall_value"].transform(lambda s: s.dropna().max() if s.notna().any() else np.nan)
        total = np.where(pd.notna(tv) & (tv > 0), tv, gsum)
    else:
        total = gsum
    df = df[["etf_ticker","name_normalized","company_ticker","value"]].copy()
    df["total_value"] = total
    df = df[df["total_value"] > 0]
    df["weight(%)"] = 100.0 * df["value"] / df["total_value"]
    df = df.groupby(["etf_ticker","name_normalized","company_ticker"], as_index=False)["weight(%)"].sum()
    df["date"] = f"{year}-12-31"
    df.rename(columns={"etf_ticker":"ETF_TICKER","name_normalized":"Normalised name","company_ticker":"Company_ticker"}, inplace=True)
    return df[["ETF_TICKER","date","Normalised name","Company_ticker","weight(%)"]]

def load_2025(path):
    df = pd.read_csv(path)
    df.columns = [c.strip().lower() for c in df.columns]
    wcol = None
    for cand in ["weight (%)","weight(%)","weight_percent","weight"]:
        if cand in df.columns:
            wcol = cand
            break
    if wcol is None:
        raise ValueError("No weight column found for 2025")
    df["weight(%)"] = df[wcol].apply(lambda x: to_float(str(x).replace("%","")))
    df = df[["etf_ticker","name_normalized","company_ticker","weight(%)"]].copy()
    df = df.groupby(["etf_ticker","name_normalized","company_ticker"], as_index=False)["weight(%)"].sum()
    df["date"] = "2025-12-31"
    df.rename(columns={"etf_ticker":"ETF_TICKER","name_normalized":"Normalised name","company_ticker":"Company_ticker"}, inplace=True)
    return df[["ETF_TICKER","date","Normalised name","Company_ticker","weight(%)"]]

past_files = sorted([p for p in PAST.glob("soi_20*_final.csv") if p.is_file()])
past_frames = [load_soi(p) for p in past_files]
cur_2025 = load_2025(BASE / "holdings_2025_final.csv")

all_holdings = pd.concat(past_frames + [cur_2025], ignore_index=True)
all_holdings["weight(%)"] = all_holdings["weight(%)"].astype(float).round(6)
all_holdings = all_holdings.sort_values(["ETF_TICKER","date","weight(%)"], ascending=[True,True,False]).reset_index(drop=True)
all_holdings.to_csv(OUT, index=False)
print(f"Wrote {OUT}")


Wrote /Users/nityaarya/Downloads/Project/blackrock-esg-etf-study/Data/Final Data/all_holdings_2017_2025.csv


In [ ]:
import pandas as pd
import yfinance as yf
import os

BASE_DIR = "/Users/nityaarya/Downloads/Project/blackrock-esg-etf-study/Data/Final Data"
HOLDINGS_2025 = os.path.join(BASE_DIR, "holdings_2025_final.csv")

OUTPUT_DIR = "/Users/nityaarya/Downloads/Project/blackrock-esg-etf-study/Data/Prices"
os.makedirs(OUTPUT_DIR, exist_ok=True)

OUT_PRICES = os.path.join(OUTPUT_DIR, "holdings_prices.csv")

# Load 2025 holdings
hold = pd.read_csv(HOLDINGS_2025, dtype=str)

# Pick normalized name column
col_name = None
for cand in ["Normalised name","name_normalized","Name","Holding Name","security"]:
    if cand in hold.columns:
        col_name = cand
        break
if col_name is None:
    raise ValueError("No normalized name column found in holdings_2025_final.csv")

# Unique company names
companies = hold[col_name].dropna().unique().tolist()

# Fetch prices using yfinance (default: last 1y daily data)
all_prices = []
for name in companies:
    try:
        ticker = yf.Ticker(name)
        hist = ticker.history(period="1y")  # can change to "5y" or "max"
        hist = hist[["Close"]].reset_index()
        hist["Company"] = name
        all_prices.append(hist)
    except Exception as e:
        print(f"Failed for {name}: {e}")

if all_prices:
    prices_df = pd.concat(all_prices, ignore_index=True)
    prices_df.to_csv(OUT_PRICES, index=False)
    print(f"Saved: {OUT_PRICES}")
else:
    print("No prices fetched")


HTTP Error 404: 
$NVIDIA: possibly delisted; no price data found  (period=1y) (Yahoo error = "No data found, symbol may be delisted")
$MICROSOFT: possibly delisted; no price data found  (period=1y)
HTTP Error 404: 
$APPLE: possibly delisted; no price data found  (period=1y) (Yahoo error = "No data found, symbol may be delisted")
Exception ignored from cffi callback <function buffer_callback at 0x14c25a480>:
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.11/site-packages/curl_cffi/curl.py", line 68, in buffer_callback
    @ffi.def_extern()
    
KeyboardInterrupt: 
$BROADCOM: possibly delisted; no price data found  (period=1y)
$ALPHABET: possibly delisted; no price data found  (period=1y) (Yahoo error = "No data found, symbol may be delisted")
$TESLA: possibly delisted; no price data found  (period=1y) (Yahoo error = "No data found, symbol may be delisted")
$HOME DEPOT: possibly delisted; no price data found  (period=1y) (Yahoo error = "No data found, symbol may be

In [6]:
import os
import pandas as pd
import numpy as np

BASE = "/Users/nityaarya/Downloads/Project/blackrock-esg-etf-study/Data/Final Data"
SRC = os.path.join(BASE, "Controversial_and_Clean_Holdings_final.xlsx")
OUT = os.path.join(BASE, "classification_binary.csv")

def read_all_sheets_xlsx(path):
    xls = pd.ExcelFile(path)
    frames = []
    for s in xls.sheet_names:
        df = pd.read_excel(xls, s)
        if isinstance(df, pd.DataFrame) and not df.empty:
            frames.append(df)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

def pick(df, *cands):
    m = {c.lower().strip(): c for c in df.columns}
    for a in cands:
        if a in m:
            return m[a]
    for a in cands:
        for k in m:
            if a in k:
                return m[k]
    return None

df = read_all_sheets_xlsx(SRC)

c_etf_tic = pick(df, "etf ticker","etf_ticker","ticker")
c_etf_name = pick(df, "etf name","name")
c_screen = pick(df, "screen category","category","screen","tag","label")
c_co_tic = pick(df, "company_ticker","company ticker","ticker")
c_nm = pick(df, "name_normalized","normalised name","normalized name","name")
c_hold_nm = pick(df, "holding name","security","security name","name","name_normalized","normalised name")

df = df.rename(columns={
    c_etf_tic: "ETF Ticker",
    c_etf_name: "ETF Name",
    c_screen: "Screen Category",
    c_co_tic: "company_ticker",
    c_nm: "name_normalized",
    c_hold_nm: "Holding Name"
})[["ETF Ticker","ETF Name","Holding Name","Screen Category","company_ticker","name_normalized"]]

for col in ["ETF Ticker","ETF Name","Holding Name","Screen Category","company_ticker","name_normalized"]:
    if col not in df.columns:
        df[col] = np.nan

df["Screen Category"] = df["Screen Category"].astype(str).str.strip()
df["company_ticker"] = df["company_ticker"].astype(str).str.strip()
df["name_normalized"] = df["name_normalized"].astype(str).str.strip()
df["Holding Name"] = df["Holding Name"].astype(str).str.strip()

def flag_contains(s, key):
    s = str(s).lower()
    return int(key in s)

df["clean200"] = df["Screen Category"].str.lower().str.contains("clean200").astype(int)
df["deforestation"] = df["Screen Category"].str.lower().str.contains("deforestation").astype(int)
df["fossil fuel"] = df["Screen Category"].str.lower().str.contains("fossil").astype(int)
df["prison"] = df["Screen Category"].str.lower().str.contains("prison").astype(int)
df["tobacco"] = df["Screen Category"].str.lower().str.contains("tobacco").astype(int)
df["weapons"] = df["Screen Category"].str.lower().str.contains("weapon").astype(int)

grp_keys = ["ETF Ticker","ETF Name","Holding Name","company_ticker","name_normalized"]

agg = (
    df.groupby(grp_keys, as_index=False)
      .agg({
          "Screen Category": lambda x: "; ".join(sorted({str(v).strip() for v in x if pd.notna(v) and str(v).strip() != ""})),
          "clean200": "max",
          "deforestation": "max",
          "fossil fuel": "max",
          "prison": "max",
          "tobacco": "max",
          "weapons": "max"
      })
)

cols = ["ETF Ticker","ETF Name","Holding Name","Screen Category","company_ticker","name_normalized",
        "clean200","deforestation","fossil fuel","prison","tobacco","weapons"]

agg = agg[cols]
agg.to_csv(OUT, index=False)
print(OUT)


/Users/nityaarya/Downloads/Project/blackrock-esg-etf-study/Data/Final Data/classification_binary.csv


In [15]:
# Analysis 1

import pandas as pd
import numpy as np
import os

BASE_DIR = "/Users/nityaarya/Downloads/Project/blackrock-esg-etf-study/Data/Final Data"
HOLDINGS_2025 = os.path.join(BASE_DIR, "holdings_2025_final.csv")
CLASS_BIN = os.path.join(BASE_DIR, "classification_binary.csv")

OUTPUT_DIR = "/Users/nityaarya/Downloads/Project/blackrock-esg-etf-study/Data/Data for Dashboard/Analysis 1"
os.makedirs(OUTPUT_DIR, exist_ok=True)

OUT_CONTEXT = os.path.join(OUTPUT_DIR, "context_summary_2025.csv")
OUT_BREAKDOWN = os.path.join(OUTPUT_DIR, "context_breakdown_by_screen.csv")

def pick_col(cols, candidates):
    cl = {c.lower(): c for c in cols}
    for cand in candidates:
        if cand.lower() in cl:
            return cl[cand.lower()]
    for c in cols:
        if c.lower().strip() in [x.lower().strip() for x in candidates]:
            return c
    return None

def to_num(x):
    if pd.isna(x):
        return np.nan
    if isinstance(x, (int, float)):
        return float(x)
    s = str(x).replace(",", "").replace("%", "").strip()
    try:
        return float(s)
    except:
        return np.nan

hold = pd.read_csv(HOLDINGS_2025, dtype=str)
cls = pd.read_csv(CLASS_BIN, dtype=str)

hold_cols = hold.columns.tolist()
cls_cols = cls.columns.tolist()

col_etf = pick_col(hold_cols, ["ETF_Ticker","ETF Ticker","etf_ticker"])
col_name = pick_col(hold_cols, ["name_normalized","Normalised name","Name","Holding Name","security"])
col_tkr = pick_col(hold_cols, ["company_ticker","Company_ticker","Ticker","company_ticker_clean"])
col_wgt = pick_col(hold_cols, ["Weight (%)","weight(%)","weight_percent","weight","Weight"])

clean_col = pick_col(cls_cols, ["clean200"])
ff_col = pick_col(cls_cols, ["fossil fuel","fossil_fuel","fossilfuel"])
weap_col = pick_col(cls_cols, ["weapons"])
tob_col = pick_col(cls_cols, ["tobacco"])
pris_col = pick_col(cls_cols, ["prison"])
defor_col = pick_col(cls_cols, ["deforestation"])
cls_tkr = pick_col(cls_cols, ["company_ticker","Company_ticker"])
cls_name = pick_col(cls_cols, ["name_normalized","Normalised name","name"])

if col_etf is None or col_wgt is None:
    raise ValueError("Missing required columns in holdings file")
if clean_col is None or any(v is None for v in [ff_col,weap_col,tob_col,pris_col,defor_col]) or (cls_tkr is None and cls_name is None):
    raise ValueError("Missing required columns in classification file")
if col_tkr is None and col_name is None:
    raise ValueError("Need either company_ticker or name_normalized in holdings")

hold["_weight"] = hold[col_wgt].apply(to_num).fillna(0.0)

cls_work = cls.copy()
for c in [clean_col, ff_col, weap_col, tob_col, pris_col, defor_col]:
    cls_work[c] = cls_work[c].astype(str).str.extract(r"(\d+)").fillna("0").astype(int)
cls_work["_any"] = (
    cls_work[ff_col] + cls_work[weap_col] + cls_work[tob_col] + cls_work[pris_col] + cls_work[defor_col]
).clip(upper=1)

if col_tkr is not None and cls_tkr is not None:
    cls_tkr_sub = cls_work[[cls_tkr, clean_col, ff_col, weap_col, tob_col, pris_col, defor_col, "_any"]].drop_duplicates(subset=[cls_tkr])
    m_tkr = hold.merge(cls_tkr_sub, left_on=col_tkr, right_on=cls_tkr, how="left")
else:
    m_tkr = hold.copy()
    for c in [clean_col, ff_col, weap_col, tob_col, pris_col, defor_col, "_any"]:
        m_tkr[c] = np.nan

if col_name is not None and cls_name is not None:
    cls_name_sub = cls_work[[cls_name, clean_col, ff_col, weap_col, tob_col, pris_col, defor_col, "_any"]].drop_duplicates(subset=[cls_name]).rename(
        columns={
            clean_col: clean_col+"_n",
            ff_col: ff_col+"_n",
            weap_col: weap_col+"_n",
            tob_col: tob_col+"_n",
            pris_col: pris_col+"_n",
            defor_col: defor_col+"_n",
            "_any": "_any_n"
        }
    )
    m_name = hold.merge(cls_name_sub, left_on=col_name, right_on=cls_name, how="left")
else:
    m_name = hold.copy()
    for c in [clean_col, ff_col, weap_col, tob_col, pris_col, defor_col, "_any"]:
        m_name[c+"_n"] = np.nan

m = m_tkr.copy()
for c in [clean_col, ff_col, weap_col, tob_col, pris_col, defor_col, "_any"]:
    m[c] = m[c].combine_first(m_name[c+"_n"])

m["_clean200"] = m[clean_col].fillna(0).astype(int)
m["_ff"] = m[ff_col].fillna(0).astype(int)
m["_weap"] = m[weap_col].fillna(0).astype(int)
m["_tob"] = m[tob_col].fillna(0).astype(int)
m["_pris"] = m[pris_col].fillna(0).astype(int)
m["_defor"] = m[defor_col].fillna(0).astype(int)
m["_any"] = ((m["_ff"] + m["_weap"] + m["_tob"] + m["_pris"] + m["_defor"]) > 0).astype(int)

is_clean = (m["_clean200"] > 0)
is_contro = (m["_any"] > 0)

m["w_clean_only"] = m["_weight"] * (is_clean & (~is_contro))
m["w_contro"] = m["_weight"] * (is_contro)  # controversy wins: includes both
m["w_other"] = m["_weight"] * ((~is_clean) & (~is_contro))

m["w_ff"] = m["_weight"] * (m["_ff"] > 0)
m["w_weap"] = m["_weight"] * (m["_weap"] > 0)
m["w_tob"] = m["_weight"] * (m["_tob"] > 0)
m["w_pris"] = m["_weight"] * (m["_pris"] > 0)
m["w_defor"] = m["_weight"] * (m["_defor"] > 0)

agg = m.groupby(col_etf, as_index=False).agg(
    total_weight=("_weight","sum"),
    clean_weight=("w_clean_only","sum"),
    controversial_weight=("w_contro","sum"),
    other_weight=("w_other","sum")
)

agg["pct_clean"] = 100.0 * agg["clean_weight"] / agg["total_weight"].replace(0, np.nan)
agg["pct_controversial"] = 100.0 * agg["controversial_weight"] / agg["total_weight"].replace(0, np.nan)
agg["pct_other"] = 100.0 * agg["other_weight"] / agg["total_weight"].replace(0, np.nan)

agg[[col_etf,"pct_clean","pct_controversial","pct_other"]].sort_values(col_etf).to_csv(OUT_CONTEXT, index=False)

screen = m.groupby(col_etf, as_index=False).agg(
    total=("_weight","sum"),
    fossil_fuel=("w_ff","sum"),
    weapons=("w_weap","sum"),
    tobacco=("w_tob","sum"),
    prison=("w_pris","sum"),
    deforestation=("w_defor","sum")
)
for c in ["fossil_fuel","weapons","tobacco","prison","deforestation"]:
    screen[c] = 100.0 * screen[c] / screen["total"].replace(0, np.nan)

screen = screen.melt(
    id_vars=[col_etf],
    value_vars=["fossil_fuel","weapons","tobacco","prison","deforestation"],
    var_name="screen",
    value_name="pct_weight"
).dropna()

screen.to_csv(OUT_BREAKDOWN, index=False)

print(f"Wrote: {OUT_CONTEXT}")
print(f"Wrote: {OUT_BREAKDOWN}")


Wrote: /Users/nityaarya/Downloads/Project/blackrock-esg-etf-study/Data/Data for Dashboard/Analysis 1/context_summary_2025.csv
Wrote: /Users/nityaarya/Downloads/Project/blackrock-esg-etf-study/Data/Data for Dashboard/Analysis 1/context_breakdown_by_screen.csv


In [14]:
import pandas as pd
import numpy as np
import os

BASE_DIR = "/Users/nityaarya/Downloads/Project/blackrock-esg-etf-study/Data/Final Data"
HOLDINGS_2025 = os.path.join(BASE_DIR, "holdings_2025_final.csv")
CLASS_BIN = os.path.join(BASE_DIR, "classification_binary.csv")

OUTPUT_DIR = "/Users/nityaarya/Downloads/Project/blackrock-esg-etf-study/Data/Data for Dashboard/Analysis 1"
os.makedirs(OUTPUT_DIR, exist_ok=True)

OUT_TOP = os.path.join(OUTPUT_DIR, "top_holdings_spotlight.csv")
TOP_N = 10

def pick_col(cols, candidates):
    cl = {c.lower(): c for c in cols}
    for cand in candidates:
        if cand.lower() in cl:
            return cl[cand.lower()]
    for c in cols:
        if c.lower().strip() in [x.lower().strip() for x in candidates]:
            return c
    return None

def to_num(x):
    if pd.isna(x):
        return np.nan
    if isinstance(x,(int,float)):
        return float(x)
    s = str(x).replace(",","").replace("%","").strip()
    try:
        return float(s)
    except:
        return np.nan

hold = pd.read_csv(HOLDINGS_2025, dtype=str)
cls = pd.read_csv(CLASS_BIN, dtype=str)

col_etf = pick_col(hold.columns, ["ETF_Ticker","ETF Ticker","etf_ticker"])
col_name = pick_col(hold.columns, ["name_normalized","Normalised name","Name","Holding Name","security"])
col_tkr = pick_col(hold.columns, ["company_ticker","Company_ticker","Ticker","company_ticker_clean"])
col_wgt = pick_col(hold.columns, ["Weight (%)","weight(%)","weight_percent","weight","Weight"])

clean_col = pick_col(cls.columns, ["clean200"])
ff_col = pick_col(cls.columns, ["fossil fuel","fossil_fuel","fossilfuel"])
weap_col = pick_col(cls.columns, ["weapons"])
tob_col = pick_col(cls.columns, ["tobacco"])
pris_col = pick_col(cls.columns, ["prison"])
defor_col = pick_col(cls.columns, ["deforestation"])
cls_tkr = pick_col(cls.columns, ["company_ticker","Company_ticker"])
cls_name = pick_col(cls.columns, ["name_normalized","Normalised name","name"])

hold["_weight"] = hold[col_wgt].apply(to_num).fillna(0.0)
if hold["_weight"].max() > 1.5:
    hold["_weight"] = hold["_weight"]/100.0

cls_w = cls.copy()
for c in [clean_col, ff_col, weap_col, tob_col, pris_col, defor_col]:
    cls_w[c] = cls_w[c].astype(str).str.extract(r"(\d+)").fillna("0").astype(int)
cls_w["_any"] = (cls_w[ff_col] + cls_w[weap_col] + cls_w[tob_col] + cls_w[pris_col] + cls_w[defor_col]).clip(upper=1)

if col_tkr and cls_tkr:
    tsub = cls_w[[cls_tkr, clean_col, ff_col, weap_col, tob_col, pris_col, defor_col, "_any"]].drop_duplicates(subset=[cls_tkr])
    m_t = hold.merge(tsub, left_on=col_tkr, right_on=cls_tkr, how="left")
else:
    m_t = hold.copy()
    for c in [clean_col, ff_col, weap_col, tob_col, pris_col, defor_col, "_any"]:
        m_t[c] = np.nan

if col_name and cls_name:
    nsub = cls_w[[cls_name, clean_col, ff_col, weap_col, tob_col, pris_col, defor_col, "_any"]].drop_duplicates(subset=[cls_name]).rename(
        columns={clean_col:f"{clean_col}_n", ff_col:f"{ff_col}_n", weap_col:f"{weap_col}_n", tob_col:f"{tob_col}_n", pris_col:f"{pris_col}_n", defor_col:f"{defor_col}_n", "_any":"_any_n"}
    )
    m_n = hold.merge(nsub, left_on=col_name, right_on=cls_name, how="left")
else:
    m_n = hold.copy()
    for c in [clean_col, ff_col, weap_col, tob_col, pris_col, defor_col, "_any"]:
        m_n[c+"_n"] = np.nan

m = m_t.copy()
for c in [clean_col, ff_col, weap_col, tob_col, pris_col, defor_col, "_any"]:
    m[c] = m[c].combine_first(m_n[c+"_n"])

m["_clean200"] = m[clean_col].fillna(0).astype(int)
m["_ff"] = m[ff_col].fillna(0).astype(int)
m["_weap"] = m[weap_col].fillna(0).astype(int)
m["_tob"] = m[tob_col].fillna(0).astype(int)
m["_pris"] = m[pris_col].fillna(0).astype(int)
m["_defor"] = m[defor_col].fillna(0).astype(int)
m["_any"] = ((m["_ff"] + m["_weap"] + m["_tob"] + m["_pris"] + m["_defor"]) > 0).astype(int)

is_clean = (m["_clean200"] > 0)
is_contro = (m["_any"] > 0)

m["category"] = np.select([is_contro, is_clean & (~is_contro)], ["controversial","clean"], default="other")

name_out = col_name if col_name is not None else col_tkr
core = m[[col_etf, name_out, col_tkr, "_weight", "category"]].rename(columns={name_out:"company_name", col_tkr:"company_ticker", "_weight":"weight"})

def top_k(g):
    out = []
    for cat in ["controversial","clean"]:
        sub = g[g["category"]==cat].sort_values("weight", ascending=False).head(TOP_N).copy()
        sub["rank"] = range(1, len(sub)+1)
        out.append(sub)
    return pd.concat(out) if len(out) else pd.DataFrame(columns=g.columns.tolist()+["rank"])

top = core.groupby(col_etf, group_keys=False).apply(top_k).reset_index(drop=True)
top["weight_pct"] = (top["weight"]*100).round(4)
top = top[[col_etf, "category", "rank", "company_name", "company_ticker", "weight_pct"]].sort_values([col_etf, "category", "rank"]).reset_index(drop=True)
top.to_csv(OUT_TOP, index=False)
print(f"Wrote: {OUT_TOP}")


Wrote: /Users/nityaarya/Downloads/Project/blackrock-esg-etf-study/Data/Data for Dashboard/Analysis 1/top_holdings_spotlight.csv


/var/folders/11/2mby0xs907bb9d9x_zhhsyn00000gn/T/ipykernel_67269/27003249.py:108: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  top = core.groupby(col_etf, group_keys=False).apply(top_k).reset_index(drop=True)


In [11]:
# Analysis 2 (controversial wins; mutually exclusive; unlabeled -> other)

import pandas as pd
import numpy as np
import os
import re

BASE_DIR = "/Users/nityaarya/Downloads/Project/blackrock-esg-etf-study/Data/Final Data"
ALL_HOLDINGS = os.path.join(BASE_DIR, "all_holdings_2017_2025.csv")
CLASS_BIN = os.path.join(BASE_DIR, "classification_binary.csv")

OUTPUT_DIR = "/Users/nityaarya/Downloads/Project/blackrock-esg-etf-study/Data/Data for Dashboard/Analysis 2"
os.makedirs(OUTPUT_DIR, exist_ok=True)

OUT_TRENDS = os.path.join(OUTPUT_DIR, "exposure_trends_2017_2025.csv")
OUT_SCREENS = os.path.join(OUTPUT_DIR, "screen_trends_by_category.csv")

def pick_col(cols, candidates):
    cl = {c.lower(): c for c in cols}
    for cand in candidates:
        if cand.lower() in cl:
            return cl[cand.lower()]
    for c in cols:
        if c.lower().strip() in [x.lower().strip() for x in candidates]:
            return c
    return None

def to_num(x):
    if pd.isna(x):
        return np.nan
    if isinstance(x, (int, float)):
        return float(x)
    s = str(x).replace(",", "").replace("%", "").strip()
    try:
        return float(s)
    except:
        return np.nan

def to_year(s):
    try:
        d = pd.to_datetime(s, errors="coerce", dayfirst=True)
        if pd.notna(d):
            return int(d.year)
    except:
        pass
    m = re.search(r"(20\d{2}|19\d{2})", str(s))
    return int(m.group(1)) if m else np.nan

hold = pd.read_csv(ALL_HOLDINGS, dtype=str)
cls = pd.read_csv(CLASS_BIN, dtype=str)

hcols = hold.columns.tolist()
ccols = cls.columns.tolist()

col_etf = pick_col(hcols, ["ETF_TICKER","ETF Ticker","etf_ticker"])
col_date = pick_col(hcols, ["date","Date"])
col_name = pick_col(hcols, ["Normalised name","name_normalized","Name","Holding Name","security"])
col_tkr = pick_col(hcols, ["Company_ticker","company_ticker","Ticker","company_ticker_clean"])
col_wgt = pick_col(hcols, ["weight(%)","Weight (%)","weight_percent","weight","Weight"])

clean_col = pick_col(ccols, ["clean200"])
ff_col = pick_col(ccols, ["fossil fuel","fossil_fuel","fossilfuel"])
weap_col = pick_col(ccols, ["weapons"])
tob_col = pick_col(ccols, ["tobacco"])
pris_col = pick_col(ccols, ["prison"])
defor_col = pick_col(ccols, ["deforestation"])
cls_tkr = pick_col(ccols, ["company_ticker","Company_ticker"])
cls_name = pick_col(ccols, ["name_normalized","Normalised name","name"])

if any(x is None for x in [col_etf, col_date, col_wgt]) or (col_tkr is None and col_name is None):
    raise ValueError("Missing required columns in all_holdings_2017_2025.csv")
if clean_col is None or any(v is None for v in [ff_col,weap_col,tob_col,pris_col,defor_col]) or (cls_tkr is None and cls_name is None):
    raise ValueError("Missing required columns in classification_binary.csv")

hold["_weight"] = hold[col_wgt].apply(to_num).fillna(0.0)
if hold["_weight"].max() > 1.5:
    hold["_weight"] = hold["_weight"] / 100.0
hold["_year"] = hold[col_date].apply(to_year)
hold = hold[pd.notna(hold["_year"])]

# Optional: ensure MPCT history is unified under SDG
if col_etf in hold.columns:
    hold[col_etf] = hold[col_etf].replace("MPCT", "SDG")

cls_work = cls.copy()
for c in [clean_col, ff_col, weap_col, tob_col, pris_col, defor_col]:
    cls_work[c] = cls_work[c].astype(str).str.extract(r"(\d+)").fillna("0").astype(int)
cls_work["_any"] = (
    cls_work[ff_col] + cls_work[weap_col] + cls_work[tob_col] + cls_work[pris_col] + cls_work[defor_col]
).clip(upper=1)

# Join by ticker then name
if col_tkr is not None and cls_tkr is not None:
    sub_tkr = cls_work[[cls_tkr, clean_col, ff_col, weap_col, tob_col, pris_col, defor_col, "_any"]].drop_duplicates(subset=[cls_tkr])
    m_tkr = hold.merge(sub_tkr, left_on=col_tkr, right_on=cls_tkr, how="left")
else:
    m_tkr = hold.copy()
    for c in [clean_col, ff_col, weap_col, tob_col, pris_col, defor_col, "_any"]:
        m_tkr[c] = np.nan

if col_name is not None and cls_name is not None:
    sub_name = cls_work[[cls_name, clean_col, ff_col, weap_col, tob_col, pris_col, defor_col, "_any"]].drop_duplicates(subset=[cls_name]).rename(
        columns={
            clean_col: clean_col+"_n",
            ff_col: ff_col+"_n",
            weap_col: weap_col+"_n",
            tob_col: tob_col+"_n",
            pris_col: pris_col+"_n",
            defor_col: defor_col+"_n",
            "_any": "_any_n"
        }
    )
    m_name = hold.merge(sub_name, left_on=col_name, right_on=cls_name, how="left")
else:
    m_name = hold.copy()
    for c in [clean_col, ff_col, weap_col, tob_col, pris_col, defor_col, "_any"]:
        m_name[c+"_n"] = np.nan

m = m_tkr.copy()
for c in [clean_col, ff_col, weap_col, tob_col, pris_col, defor_col, "_any"]:
    m[c] = m[c].combine_first(m_name[c+"_n"])

m["_clean200"] = m[clean_col].fillna(0).astype(int)
m["_ff"] = m[ff_col].fillna(0).astype(int)
m["_weap"] = m[weap_col].fillna(0).astype(int)
m["_tob"] = m[tob_col].fillna(0).astype(int)
m["_pris"] = m[pris_col].fillna(0).astype(int)
m["_defor"] = m[defor_col].fillna(0).astype(int)
m["_any"] = ((m["_ff"] + m["_weap"] + m["_tob"] + m["_pris"] + m["_defor"]) > 0).astype(int)

# Controversy wins: mutually exclusive
is_clean = (m["_clean200"] > 0)
is_contro = (m["_any"] > 0)

m["w_clean_only"] = m["_weight"] * (is_clean & (~is_contro))
m["w_contro"]      = m["_weight"] * (is_contro)
m["w_other"]       = m["_weight"] * ((~is_clean) & (~is_contro))

m["w_ff"]   = m["_weight"] * (m["_ff"] > 0)
m["w_weap"] = m["_weight"] * (m["_weap"] > 0)
m["w_tob"]  = m["_weight"] * (m["_tob"] > 0)
m["w_pris"] = m["_weight"] * (m["_pris"] > 0)
m["w_defor"]= m["_weight"] * (m["_defor"] > 0)

agg = (
    m.groupby([col_etf, "_year"], as_index=False)
     .agg(total_weight=("_weight","sum"),
          clean_weight=("w_clean_only","sum"),
          controversial_weight=("w_contro","sum"),
          other_weight=("w_other","sum"))
)

agg["pct_clean"] = 100.0 * agg["clean_weight"] / agg["total_weight"].replace(0, np.nan)
agg["pct_controversial"] = 100.0 * agg["controversial_weight"] / agg["total_weight"].replace(0, np.nan)
agg["pct_other"] = 100.0 * agg["other_weight"] / agg["total_weight"].replace(0, np.nan)

exposure_trends = (
    agg[[col_etf, "_year", "pct_clean", "pct_controversial", "pct_other"]]
    .rename(columns={"_year":"year"})
    .sort_values([col_etf, "year"])
    .reset_index(drop=True)
)
exposure_trends.to_csv(OUT_TRENDS, index=False)

scr = (
    m.groupby([col_etf, "_year"], as_index=False)
     .agg(total=("_weight","sum"),
          fossil_fuel=("w_ff","sum"),
          weapons=("w_weap","sum"),
          tobacco=("w_tob","sum"),
          prison=("w_pris","sum"),
          deforestation=("w_defor","sum"))
)

for c in ["fossil_fuel","weapons","tobacco","prison","deforestation"]:
    scr[c] = 100.0 * scr[c] / scr["total"].replace(0, np.nan)

scr_long = (
    scr.melt(
        id_vars=[col_etf, "_year"],
        value_vars=["fossil_fuel","weapons","tobacco","prison","deforestation"],
        var_name="screen",
        value_name="pct_weight"
    )
    .rename(columns={"_year":"year"})
    .sort_values([col_etf, "year", "screen"])
    .reset_index(drop=True)
)
scr_long.to_csv(OUT_SCREENS, index=False)

print(f"Wrote: {OUT_TRENDS}")
print(f"Wrote: {OUT_SCREENS}")


/var/folders/11/2mby0xs907bb9d9x_zhhsyn00000gn/T/ipykernel_67269/540370380.py:41: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  d = pd.to_datetime(s, errors="coerce", dayfirst=True)


Wrote: /Users/nityaarya/Downloads/Project/blackrock-esg-etf-study/Data/Data for Dashboard/Analysis 2/exposure_trends_2017_2025.csv
Wrote: /Users/nityaarya/Downloads/Project/blackrock-esg-etf-study/Data/Data for Dashboard/Analysis 2/screen_trends_by_category.csv


In [12]:
# Analysis 2 (AUM-weighted aggregates; consistent with controversy-wins policy)

import pandas as pd
import numpy as np
import os
import glob
import re

BASE_DIR = "/Users/nityaarya/Downloads/Project/blackrock-esg-etf-study/Data/Final Data"
PAST_DIR = os.path.join(BASE_DIR, "Combined Past Holdings")
HOLDINGS_2025 = os.path.join(BASE_DIR, "holdings_2025_final.csv")

OUTPUT_DIR = "/Users/nityaarya/Downloads/Project/blackrock-esg-etf-study/Data/Data for Dashboard/Analysis 2"
os.makedirs(OUTPUT_DIR, exist_ok=True)

EXPOSURE_TRENDS = os.path.join(OUTPUT_DIR, "exposure_trends_2017_2025.csv")
SCREEN_TRENDS = os.path.join(OUTPUT_DIR, "screen_trends_by_category.csv")

OUT_AUM   = os.path.join(OUTPUT_DIR, "etf_aum_by_year.csv")
OUT_AGG_EXP = os.path.join(OUTPUT_DIR, "aggregate_exposure_trends.csv")
OUT_AGG_SCR = os.path.join(OUTPUT_DIR, "aggregate_screen_trends.csv")
OUT_DISP    = os.path.join(OUTPUT_DIR, "exposure_dispersion_stats.csv")

YEAR_MIN, YEAR_MAX = 2017, 2025

def pick_col(cols, candidates):
    cl = {c.lower(): c for c in cols}
    for cand in candidates:
        if cand.lower() in cl:
            return cl[cand.lower()]
    for c in cols:
        if c.lower().strip() in [x.lower().strip() for x in candidates]:
            return c
    return None

def to_num(x):
    if pd.isna(x):
        return np.nan
    if isinstance(x,(int,float)):
        return float(x)
    s = str(x).replace(",","").replace("%","").replace("₹","").replace("$","").replace("€","").replace("£","").strip()
    try:
        return float(s)
    except:
        return np.nan

def to_year(s):
    try:
        d = pd.to_datetime(s, errors="coerce", dayfirst=True)
        if pd.notna(d):
            y = int(d.year)
            if YEAR_MIN-10 <= y <= YEAR_MAX:
                return y
    except:
        pass
    m = re.search(r"(20\d{2}|19\d{2})", str(s))
    if m:
        y = int(m.group(1))
        if YEAR_MIN-10 <= y <= YEAR_MAX:
            return y
    return np.nan

def extract_aum_from_soi(path, year_hint=None):
    df = pd.read_csv(path, dtype=str)
    etf_col = pick_col(df.columns, ["etf_ticker","ETF_Ticker","ETF Ticker"])
    date_col = pick_col(df.columns, ["date","Date"])
    val_col  = pick_col(df.columns, ["value","holding_value","market_value","Value","Market Value"])
    tot_val_col = pick_col(df.columns, ["total_value","Total Value","fund_total_value"])
    tot_overall_col = pick_col(df.columns, ["total_overall_value","Total Overall Value","fund_total_overall_value","fund_total"])
    if etf_col is None:
        return pd.DataFrame(columns=["ETF_TICKER","year","AUM"])

    df["_etf"] = df[etf_col].astype(str).replace("MPCT","SDG")
    df["_year"] = df[date_col].apply(to_year) if date_col is not None else year_hint

    for c in [val_col, tot_val_col, tot_overall_col]:
        if c is not None:
            df[c] = df[c].apply(to_num)

    parts = []
    for (e,y), g in df.groupby(["_etf","_year"]):
        if pd.isna(y): continue
        aum = np.nan
        if tot_overall_col is not None and g[tot_overall_col].notna().any():
            aum = g[tot_overall_col].dropna().iloc[-1]
        if (np.isnan(aum) or aum == 0) and tot_val_col is not None and g[tot_val_col].notna().any():
            aum = g[tot_val_col].dropna().iloc[-1]
        if (np.isnan(aum) or aum == 0) and val_col is not None:
            aum = g[val_col].sum()
        parts.append({"ETF_TICKER": e, "year": int(y), "AUM": float(aum if pd.notna(aum) else 0.0)})

    out = pd.DataFrame(parts)
    if not out.empty:
        out = out[(out["year"] >= YEAR_MIN) & (out["year"] <= YEAR_MAX)]
    return out

def extract_aum_2025(path):
    df = pd.read_csv(path, dtype=str)
    etf_col = pick_col(df.columns, ["ETF_Ticker","ETF Ticker","etf_ticker"])
    tot_overall_col = pick_col(df.columns, ["total_overall_value","Total Overall Value"])
    tot_val_col     = pick_col(df.columns, ["total_value","Total Value"])
    mval_col        = pick_col(df.columns, ["Market Value","market_value","value"])
    if etf_col is None:
        return pd.DataFrame(columns=["ETF_TICKER","year","AUM"])

    df["_etf"] = df[etf_col].astype(str).replace("MPCT","SDG")
    for c in [tot_overall_col, tot_val_col, mval_col]:
        if c is not None:
            df[c] = df[c].apply(to_num)

    parts = []
    for e, g in df.groupby("_etf"):
        aum = np.nan
        if tot_overall_col is not None and g[tot_overall_col].notna().any():
            aum = g[tot_overall_col].dropna().iloc[-1]
        if (np.isnan(aum) or aum == 0) and tot_val_col is not None and g[tot_val_col].notna().any():
            aum = g[tot_val_col].dropna().iloc[-1]
        if (np.isnan(aum) or aum == 0) and mval_col is not None:
            aum = g[mval_col].sum()
        parts.append({"ETF_TICKER": e, "year": 2025, "AUM": float(aum if pd.notna(aum) else 0.0)})

    out = pd.DataFrame(parts)
    out = out[(out["year"] >= YEAR_MIN) & (out["year"] <= YEAR_MAX)]
    return out

soi_paths = sorted(glob.glob(os.path.join(PAST_DIR, "soi_20*_final.csv")))
year_from_name = []
for p in soi_paths:
    b = os.path.basename(p)
    yr = None
    for tok in re.findall(r"\d{4}", b):
        if tok.isdigit():
            yr = int(tok)
    year_from_name.append((p, yr))

aum_frames = [extract_aum_from_soi(p, yr) for p,yr in year_from_name]
aum_frames.append(extract_aum_2025(HOLDINGS_2025))

aum_by_year = pd.concat(aum_frames, ignore_index=True) if aum_frames else pd.DataFrame(columns=["ETF_TICKER","year","AUM"])
aum_by_year = aum_by_year.dropna(subset=["year"])
aum_by_year["year"] = aum_by_year["year"].astype(int)
aum_by_year = aum_by_year[(aum_by_year["year"] >= YEAR_MIN) & (aum_by_year["year"] <= YEAR_MAX)]
aum_by_year = aum_by_year.groupby(["ETF_TICKER","year"], as_index=False)["AUM"].sum()
aum_by_year.to_csv(OUT_AUM, index=False)

exp = pd.read_csv(EXPOSURE_TRENDS)
col_etf = pick_col(exp.columns, ["ETF_TICKER","ETF Ticker","etf_ticker"])
col_year = pick_col(exp.columns, ["year","Year"])
exp = exp.rename(columns={col_etf:"ETF_TICKER", col_year:"year"})
for c in ["pct_clean","pct_controversial","pct_other"]:
    exp[c] = exp[c].apply(to_num)
exp["ETF_TICKER"] = exp["ETF_TICKER"].astype(str).replace("MPCT","SDG")
exp = exp[(exp["year"] >= YEAR_MIN) & (exp["year"] <= YEAR_MAX)]

scr = pd.read_csv(SCREEN_TRENDS)
col_etf2 = pick_col(scr.columns, ["ETF_TICKER","ETF Ticker","etf_ticker"])
col_year2 = pick_col(scr.columns, ["year","Year"])
scr = scr.rename(columns={col_etf2:"ETF_TICKER", col_year2:"year"})
scr["pct_weight"] = scr["pct_weight"].apply(to_num)
scr["ETF_TICKER"] = scr["ETF_TICKER"].astype(str).replace("MPCT","SDG")
scr = scr[(scr["year"] >= YEAR_MIN) & (scr["year"] <= YEAR_MAX)]

exp_w = exp.merge(aum_by_year, on=["ETF_TICKER","year"], how="left")
exp_w["AUM"] = exp_w["AUM"].fillna(0.0)

def wavg(series, weights):
    w = np.asarray(weights)
    x = np.asarray(series)
    s = np.nansum(w)
    if s <= 0:
        return np.nan
    return float(np.nansum(x * w) / s)

agg_exp = (
    exp_w.groupby("year", as_index=False)
         .apply(lambda g: pd.Series({
             "total_aum": g["AUM"].sum(),
             "pct_clean_w": wavg(g["pct_clean"], g["AUM"]),
             "pct_controversial_w": wavg(g["pct_controversial"], g["AUM"]),
             "pct_other_w": wavg(g["pct_other"], g["AUM"])
         }))
         .reset_index(drop=True)
         .sort_values("year")
)
agg_exp.to_csv(OUT_AGG_EXP, index=False)

scr_w = scr.merge(aum_by_year, on=["ETF_TICKER","year"], how="left")
scr_w["AUM"] = scr_w["AUM"].fillna(0.0)

agg_scr = (
    scr_w.groupby(["year","screen"], as_index=False)
         .apply(lambda g: pd.Series({
             "total_aum": g["AUM"].sum(),
             "pct_weight_w": wavg(g["pct_weight"], g["AUM"])
         }))
         .reset_index(drop=True)
         .sort_values(["year","screen"])
)
agg_scr.to_csv(OUT_AGG_SCR, index=False)

disp = (
    exp.groupby("year").agg(
        pct_clean_min=("pct_clean","min"),
        pct_clean_max=("pct_clean","max"),
        pct_clean_mean=("pct_clean","mean"),
        pct_clean_median=("pct_clean","median"),
        pct_clean_std=("pct_clean","std"),

        pct_cont_min=("pct_controversial","min"),
        pct_cont_max=("pct_controversial","max"),
        pct_cont_mean=("pct_controversial","mean"),
        pct_cont_median=("pct_controversial","median"),
        pct_cont_std=("pct_controversial","std"),

        pct_other_min=("pct_other","min"),
        pct_other_max=("pct_other","max"),
        pct_other_mean=("pct_other","mean"),
        pct_other_median=("pct_other","median"),
        pct_other_std=("pct_other","std"),

        etf_count=("pct_clean","count")
    )
    .reset_index()
    .sort_values("year")
)
disp.to_csv(OUT_DISP, index=False)

print(f"Wrote: {OUT_AUM}")
print(f"Wrote: {OUT_AGG_EXP}")
print(f"Wrote: {OUT_AGG_SCR}")
print(f"Wrote: {OUT_DISP}")


Wrote: /Users/nityaarya/Downloads/Project/blackrock-esg-etf-study/Data/Data for Dashboard/Analysis 2/etf_aum_by_year.csv
Wrote: /Users/nityaarya/Downloads/Project/blackrock-esg-etf-study/Data/Data for Dashboard/Analysis 2/aggregate_exposure_trends.csv
Wrote: /Users/nityaarya/Downloads/Project/blackrock-esg-etf-study/Data/Data for Dashboard/Analysis 2/aggregate_screen_trends.csv
Wrote: /Users/nityaarya/Downloads/Project/blackrock-esg-etf-study/Data/Data for Dashboard/Analysis 2/exposure_dispersion_stats.csv


/var/folders/11/2mby0xs907bb9d9x_zhhsyn00000gn/T/ipykernel_67269/3715222236.py:176: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: pd.Series({
/var/folders/11/2mby0xs907bb9d9x_zhhsyn00000gn/T/ipykernel_67269/3715222236.py:192: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: pd.Series({
